# Local: arXiv rồi S2, resume cache cũ
Mở repo bằng VS Code, chọn kernel `.venv/Scripts/python.exe` (cài ipykernel nếu VS Code yêu cầu).
Dừng crawler khác trước khi Run All. Đọc CONTACT_EMAIL và S2_API_KEY từ .env, không ghi key vào notebook.
Giữ toàn Range: tự bỏ qua phần hợp lệ, không bắt đầu cứng từ bài thứ 2500.
Interrupt cell để dừng; không tắt kernel/đóng máy trước khi bước export hoàn tất.


In [ ]:
from pathlib import Path
import sys
from dataclasses import replace

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'scripts/crawl_local.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Mở notebook bên trong repo arxiv-reference-matching')
sys.path.insert(0, str(REPO))
from src.common.config import load_env, get_settings
from scripts.crawl_range import parse_range
from scripts.crawl_local import cache_roots, run_local
load_env(REPO / '.env')
settings = get_settings()
IDS = list(parse_range('2412.08848', '2412.13847').ids())
OUTPUT = REPO / 'data/local-sequential'
CACHES = cache_roots(REPO / 'data')
print('Cache đầu vào:', *CACHES, sep='\n')
print('Output:', OUTPUT)


In [ ]:
# Chạy tuần tự; không gọi S2 nếu đã Interrupt ở bước arXiv.
# Mỗi dịch vụ chạy một lượt. Lỗi còn thiếu sẽ được thử lại lần Run All sau.
reports = run_local(settings, IDS, roots=CACHES, output=OUTPUT,
                    stage='all', max_passes=1)
for stage, report in reports.items():
    print(stage, {k: v for k, v in report.items() if k != 'pending_ids'})


## Kết quả
- `data/local-sequential/arxiv/tables/papers.parquet`: riêng arXiv.
- `data/local-sequential/s2/tables/references.parquet`: riêng S2.
- `data/local-sequential/merged/tables/`: hai bảng cuối theo điều kiện hoàn tất cũ.
- `merged/manifests/merge-report.json`: danh sách Publication còn thiếu.
Chạy lại Run All để resume. Cache nằm trên ổ đĩa local, không cần Save Version hay nén ZIP.
Lần đầu import/checksum có thể lâu. Bài đã đủ không gọi mạng lại.
